slict2 LC

In [9]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

from evo.core import metrics
import evo.core.sync as sync
import evo.core.trajectory as evotraj

# Paths to the directories containing estimate and ground truth
EST_DIR = '/home/dat/slict_ws/src/slict/evo/mcdviral_ntu/slict2_lc'
GRT_DIR = '/home/dat/slict_ws/src/slict/evo/mcdviral_ntu/grt'


def load_estimate(est_path):
    """
    Load an estimated trajectory from a PCD file.
    - Skips the first 11 header lines.
    - Assumes columns: x y z intensity timestamp qx qy qz qw
    """
    data = np.loadtxt(est_path, delimiter=' ', skiprows=11)
    timestamps = data[:, 4]
    positions = data[:, 0:3]
    # Reorder quaternion to w,x,y,z
    quaternions = data[:, [8, 5, 6, 7]]
    return evotraj.PoseTrajectory3D(
        positions_xyz=positions,
        orientations_quat_wxyz=quaternions,
        timestamps=timestamps
    )


def load_groundtruth(grt_path):
    """
    Load a ground truth trajectory from a CSV file.
    - Skips the header line.
    - Assumes columns:
        col 1: timestamp,
        cols 2-4: x, y, z,
        cols 5-8: qx, qy, qz, qw  (reorder to w, x, y, z)
    """
    data = np.loadtxt(grt_path, delimiter=',', skiprows=1)
    timestamps = data[:, 1]
    positions = data[:, [2, 3, 4]]
    quaternions = data[:, [8, 5, 6, 7]]
    return evotraj.PoseTrajectory3D(
        positions_xyz=positions,
        orientations_quat_wxyz=quaternions,
        timestamps=timestamps
    )


def associate_and_align(traj_est, traj_gtr, max_diff=0.2):
    """
    Sync trajectories by timestamp then align estimate to groundtruth.
    Returns: (est_sync_aligned, gtr_sync)
    """
    est_sync, gtr_sync = sync.associate_trajectories(
        traj_est, traj_gtr, max_diff=max_diff
    )

    # Align estimate onto groundtruth frame (SE(3) alignment)
    est_sync.align(gtr_sync)

    return est_sync, gtr_sync


def compute_rmse_aligned(gtr_sync, est_sync_aligned):
    """
    Compute translation RMSE after alignment
    """
    ape_metric = metrics.APE(pose_relation=metrics.PoseRelation.translation_part)
    ape_metric.process_data((gtr_sync, est_sync_aligned))
    rmse = float(
        ape_metric
        .get_result(ref_name='reference', est_name='estimate')
        .stats['rmse']
    )
    return rmse


def process_all_sets(max_diff=0.5, do_plot=False):
    # Find common subfolders in EST_DIR and GRT_DIR
    est_sets = [d for d in os.listdir(EST_DIR) if os.path.isdir(os.path.join(EST_DIR, d))]
    grt_sets = [d for d in os.listdir(GRT_DIR) if os.path.isdir(os.path.join(GRT_DIR, d))]
    common_sets = sorted(set(est_sets) & set(grt_sets))

    results = {}

    for subset in common_sets:
        est_files = glob.glob(os.path.join(EST_DIR, subset, '*.pcd'))
        grt_files = glob.glob(os.path.join(GRT_DIR, subset, '*.csv'))

        if not est_files or not grt_files:
            print(f"[WARN] Skipping {subset}: missing .pcd or .csv file")
            continue

        est_path = est_files[0]
        grt_path = grt_files[0]

        print(f"Processing set {subset}...")

        traj_est = load_estimate(est_path)
        traj_gtr = load_groundtruth(grt_path)

        # ✅ sync + align đúng
        est_aligned, gtr_sync = associate_and_align(traj_est, traj_gtr, max_diff=max_diff)

        # ✅ RMSE sau align
        rmse = compute_rmse_aligned(gtr_sync, est_aligned)
        results[subset] = rmse

        # ✅ plot đúng dữ liệu đã sync + align
        if do_plot:
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.plot(
                est_aligned.positions_xyz[:, 0],
                est_aligned.positions_xyz[:, 1],
                '-', linewidth=2, label='estimate (aligned)'
            )
            ax.plot(
                gtr_sync.positions_xyz[:, 0],
                gtr_sync.positions_xyz[:, 1],
                '--', linewidth=2, label='ground truth'
            )
            ax.set_aspect('equal')
            ax.set_title(f"{subset} — RMSE = {rmse:.3f} m")
            ax.set_xlabel('X [m]')
            ax.set_ylabel('Y [m]')
            ax.legend()
            ax.grid(True)
            plt.show()

    print("\nSummary of RMSE for all processed sets:")
    for subset, rmse_val in results.items():
        print(f"  {subset}: {rmse_val:.3f} m")


if __name__ == '__main__':
    process_all_sets(max_diff=0.2, do_plot=False)


Processing set ntu_day_01...
Processing set ntu_day_02...
Processing set ntu_day_03...
Processing set ntu_day_04...
Processing set ntu_day_05...
Processing set ntu_day_06...
Processing set ntu_day_07...
Processing set ntu_day_08...
Processing set ntu_day_09...
Processing set ntu_day_10...
Processing set ntu_night_01...
Processing set ntu_night_02...
Processing set ntu_night_03...
Processing set ntu_night_04...
Processing set ntu_night_05...
Processing set ntu_night_06...
Processing set ntu_night_07...
Processing set ntu_night_08...
Processing set ntu_night_09...
Processing set ntu_night_10...
Processing set ntu_night_11...
Processing set ntu_night_12...
Processing set ntu_night_13...

Summary of RMSE for all processed sets:
  ntu_day_01: 1.453 m
  ntu_day_02: 0.498 m
  ntu_day_03: 1.817 m
  ntu_day_04: 2.490 m
  ntu_day_05: 1.689 m
  ntu_day_06: 1.671 m
  ntu_day_07: 1.415 m
  ntu_day_08: 2.002 m
  ntu_day_09: 2.388 m
  ntu_day_10: 1.432 m
  ntu_night_01: 1.431 m
  ntu_night_02: 1.345 